# TrialSpec-LM: Kaggle training runner

Thin wrapper: all logic lives in the GitHub repo (`src/`). This notebook clones it and runs the scripts.

**Settings:** Accelerator = GPU T4 x2 (or P100), Internet = On. Secrets: `WANDB_API_KEY`, `HF_TOKEN`.

**Order:** run the SMOKE cells first. Only set `RUN_FULL = True` after the smoke test prints `SMOKE TEST PASSED`, then use *Save Version → Save & Run All* so training runs in the background.

In [ ]:
RUN_FULL = False      # set True for the real training run
RUN_BASELINE = True   # zero-shot 'before' predictions on val
BRANCH = 'master'

In [ ]:
import os, subprocess
REPO = '/kaggle/working/TrialSpec-LM'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/Umairishuman/TrialSpec-LM.git', REPO], check=True)
os.chdir(REPO)
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['WANDB_API_KEY'] = s.get_secret('WANDB_API_KEY')
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # single GPU: avoids DataParallel with LoRA (see decisions log)
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__)
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')

In [ ]:
# Sanity: split must match the committed manifest; no GPT-4 leakage.
!python -m pytest -q tests

## 1. Smoke tests (minutes)

In [ ]:
!python -m src.train.run --smoke
!python -m src.eval.predict --smoke --split val --out results/smoke/baseline_val

## 2. Zero-shot baseline (before fine-tuning) on val

In [ ]:
if RUN_BASELINE:
    !python -m src.eval.predict --split val --out results/baseline/qwen2.5-1.5b_val

## 3. Full fine-tune + fine-tuned predictions on val

In [ ]:
if RUN_FULL:
    !python -m src.train.run --config configs/train.yaml
    !python -m src.eval.predict --split val --adapter outputs/qwen2.5-1.5b-lora/final --out results/finetuned/qwen2.5-1.5b-lora_val

In [ ]:
# Everything under /kaggle/working is kept as notebook output; zip results for download.
!cd /kaggle/working && zip -qr results.zip TrialSpec-LM/results TrialSpec-LM/outputs/*/final 2>/dev/null; ls -la /kaggle/working